# Leakage audit

Issue: #12

## Question

Does any future information reach training? For every place it could, we write a check that would fail if it did, and run it on the real data.

## Data used

`data/sample/events.parquet` (#4), `data/covis/*` (#7), `data/categories/*` (#8), `data/eval/ranking_queries.parquet` (#10), and the window definitions in `notebooks/split.py` (#6). Window checked: **w0** (train days 0-13, eval days 14-15).

The result is summarised in `docs/leakage.md`. Re-run this notebook after every task that adds a feature or a count (see the last section).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from split import WINDOWS, day_to_ms, split_window

DATA = Path("data") if Path("data").exists() else Path("../data")
W = WINDOWS["w0 initial (weeks 1-2)"]
TRAIN_END, EVAL_START, EVAL_END = (day_to_ms(W[k]) for k in ("train_end_day", "eval_start_day", "eval_end_day"))
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
results = []   # (place, status, evidence)


def record(place, status, evidence):
    results.append({"place": place, "status": status, "evidence": evidence})
    print(f"[{status}] {place}: {evidence}")

## A reusable check: are these counts really from the training window?

Anything computed from event counts (popularity, item frequency, co-visitation item sets, rates) can be tested by recomputing the counts from `ts < train_end` and comparing.

In [ ]:
def counts_from_window(lf, train_end):
    return dict(lf.filter(pl.col("ts") < train_end).group_by("aid").len().collect().iter_rows())


def check_counts_from_window(counts, lf, train_end, name):
    """Fails if `counts` (dict aid -> n) is not exactly the count from events before train_end."""
    expected = counts_from_window(lf, train_end)
    assert counts == expected, f"{name}: counts are not from the training window only"


train_counts = counts_from_window(events, TRAIN_END)
check_counts_from_window(train_counts, events, TRAIN_END, "train counts")   # passes trivially

# The deliberate leak: counts over ALL four weeks must fail the check.
all_counts = dict(events.group_by("aid").len().collect().iter_rows())
try:
    check_counts_from_window(all_counts, events, TRAIN_END, "popularity over all weeks")
except AssertionError as err:
    print("deliberate leak caught:", err)
else:
    raise AssertionError("the check did not catch counts over all weeks")

## 1. Split boundaries

In [ ]:
train, evaluation = split_window(events, "w0 initial (weeks 1-2)")
train_max = train.select(pl.col("ts").max()).collect()[0, 0]
eval_stats = evaluation.select(pl.col("ts").min().alias("mn"), pl.col("ts").max().alias("mx")).collect()
assert train_max < eval_stats["mn"][0]
assert eval_stats["mn"][0] >= EVAL_START and eval_stats["mx"][0] < EVAL_END
overlap = train.select("session").unique().join(evaluation.select("session").unique(), on="session").select(pl.len()).collect()[0, 0]
assert overlap == 0
record("Split boundaries", "checked automatically",
       f"last train event {train_max} < first eval event {eval_stats['mn'][0]}; eval inside [{EVAL_START}, {EVAL_END}); {overlap} sessions in both")

## 2. Co-visitation matrix

If eval-window events had been counted, the set of items with at least 5 events would differ from the set in the matrix.

In [ ]:
meta = json.loads((DATA / "covis" / "meta.json").read_text())
item_ids = np.load(DATA / "covis" / "item_ids.npy")
expected_items = sorted(a for a, n in train_counts.items() if n >= meta["min_freq"])
assert meta["train_end_ms"] == TRAIN_END
assert item_ids.tolist() == expected_items
# Items that would only qualify if the eval days were counted:
counts_with_eval = dict(events.filter(pl.col("ts") < EVAL_END).group_by("aid").len().collect().iter_rows())
would_leak = sum(1 for a, n in counts_with_eval.items() if n >= meta["min_freq"] and train_counts.get(a, 0) < meta["min_freq"])
assert would_leak > 0, "the check could not tell the difference"
record("Co-visitation matrix", "checked automatically",
       f"item set equals items with >= {meta['min_freq']} events before the train end ({len(item_ids):,} items); {would_leak:,} items would differ if eval days were counted")

## 3. Categories

In [ ]:
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
assert set(categories["aid"].to_list()) <= set(item_ids.tolist())
record("Categories", "checked automatically",
       f"{categories.height:,} categorised items, all inside the co-visitation item set (which is training window only)")

## 4. Query construction

The query category is taken from the held-out click. That is a **known leak** (see #10), accepted on purpose. What must not leak is time order: the prefix must come from the eval window and end before the labels.

In [ ]:
q = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
assert (q["label_ts_first"] >= q["prefix_ts_last"]).all()
assert (q["prefix_ts_last"] < EVAL_END).all()
firsts = events.group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
qs = q.select("session").join(firsts, on="session")
assert qs.height == q.height and (qs["first_ts"] >= EVAL_START).all() and (qs["first_ts"] < EVAL_END).all()

# integrity: prefix and labels match the real events for 300 random queries
sample = q.sample(300, seed=0)
real = evaluation.filter(pl.col("session").is_in(sample["session"].to_list())).sort("session", "ts").collect()
by_session = {k[0]: v for k, v in real.group_by("session", maintain_order=True)}
for row in sample.iter_rows(named=True):
    ev = by_session[row["session"]]["aid"].to_list()
    p, lab = row["prefix_aid"], row["label_aid"]
    assert ev[: len(p)] == p and ev[len(p): len(p) + len(lab)] == lab
record("Query construction: time order", "checked automatically",
       "all query sessions start in the eval window; prefix ends before labels; 300 random queries match the raw events")
record("Query construction: category from the label", "known leak (accepted)",
       "the query category is the category of the held-out click, see #10; #36 measures the effect of predicted categories")

## 5. Popularity, rates and other counts (used from #17 on)

Nothing computes them yet. The helper above must be called with the counts used by every ranker.

In [ ]:
record("Popularity and rate features", "pending: re-run after #17 and #22",
       "call check_counts_from_window on the counts each baseline or feature uses; the deliberate-leak demo above shows it fails on all-weeks counts")

## 6. Session sampling

In [ ]:
manifest = json.loads((DATA / "sample" / "sample_manifest.json").read_text())
record("Session sampling", "not applicable (no outcome used)",
       f"sessions are chosen by a hash of the session ID only ({manifest['keep_pct']} percent); the rule never looks at events or labels")

## 7. Items the training window has never seen

In [ ]:
train_items = set(train_counts)
label_items = q.select(pl.col("label_aid").explode().alias("aid"))
total = label_items.height
cold = label_items.filter(~pl.col("aid").is_in(list(train_items))).height
uncat = q.select(pl.col("label_cluster").explode().alias("c")).filter(pl.col("c") < 0).height
record("Items never seen in training", "not a leak, a coverage limit",
       f"{cold:,} of {total:,} label events ({cold / total:.1%}) are on items with no training events; {uncat:,} ({uncat / total:.1%}) are on items without a category. Models cannot retrieve them")

## 8. Sessions crossing the boundary

In [ ]:
spans = events.group_by("session").agg(pl.col("ts").min().alias("first"), pl.col("ts").max().alias("last")).collect()
cross = spans.filter((pl.col("first") < TRAIN_END) & (pl.col("last") >= TRAIN_END))
train_sessions = set(train.select("session").unique().collect()["session"].to_list())
eval_sessions = set(evaluation.select("session").unique().collect()["session"].to_list())
assert not (eval_sessions & set(cross["session"].to_list()))
record("Sessions crossing the boundary", "checked automatically",
       f"{cross.height:,} sessions cross the train end; their post-boundary events are dropped from train and none of them is an eval session")

## Summary

In [ ]:
summary = pl.DataFrame(results)
with pl.Config(fmt_str_lengths=200, tbl_rows=20, tbl_width_chars=200):
    print(summary)
auto = [r for r in results if r["status"] == "checked automatically"]
assert len(results) >= 6 and len(auto) >= 4
print(f"{len(results)} places listed, {len(auto)} checked automatically")

## Where to re-run

Re-run this notebook, and add the check for the new code, after: #17 (popularity baseline, call `check_counts_from_window`), #18 (co-visitation scoring), #22 (features as of a cutoff), #23 to #28 (any training set built from sessions), #36 (predicted categories).

## What I learned

To be written by the owner of the project.